## Sampling ERA5 hourly met data over arbitrary geometries and exporting to ELM formats
This notebook demonstrates how to sample raw ERA5-Land hourly meteorological data from Google Earth Engine using either points or polygons, then process the downloaded data for export to ELM-ready netCDF files. Understanding this example will likely cover your ERA5 use-case. We will cover two different ways to export your data.
  
This example considers 1x1 degree polygons over 11 sites (coordinates) of interest to NGEE-Arctic. While these polygons are "regular" in the sense they're all the same size and shape, the workflow would look the same for arbitrary polygons, and you could run with just the point geometries if you wish.

Please note that while we call this data source "ERA5", it specifically refers to ERA5-Land hourly hosted on Google Earth Engine. The details for this dataset can be found [here](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_HOURLY).

Also please note that `dapper` is currently only configured for preparing met data for OLMT's *coupler bypass* mode, not *DATM*. If you need *DATM*, please open an issue.

We'll start by generating our geometries in a `GeoDataFrame`.

In [1]:
import ee
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Polygon

from dapper.utils import utils
from dapper.utils import gee_utils as gu
from dapper.config.metsources import era5
from dapper.met.adapters.era5 import ERA5Adapter
from dapper.met.exporter import Exporter

# Make sure to Initialize with the correct project name (do not use mine--it won't work for you)
ee.Initialize(project='ee-jonschwenk')

# Build our points dictionary
sites = {'tl27' : (-165.959, 64.735),
    'tl47' :  (-166.12778, 65.01896),
    'kg' : (-164.82771, 65.16185),
    'kfc' : (-164.65123, 65.45158),
    'cnl' : (-163.7181, 64.8541),
    'utq' :  (-156.65414, 71.31381),
    'tvc' : (-133.49919, 68.74207),
    'tfs' : (-149.59351, 68.62758),
    'abs' : (18.81545, 68.35418),
    'bs' : (11.835, 78.9325),
    'si' : (126.47446, 72.37004)}

# Create 1x1 degree box geometries around each point
geometries = []
gids = []
half_size = 0.5  # degrees

for gid, (lon, lat) in sites.items(): 
    # sanity checks
    assert -180 <= lon <= 180, f"{gid}: lon out of range: {lon}"
    assert -90  <= lat <= 90,  f"{gid}: lat out of range: {lat}"

    box = Polygon([
        (lon - half_size, lat - half_size),
        (lon + half_size, lat - half_size),
        (lon + half_size, lat + half_size),
        (lon - half_size, lat + half_size),
        (lon - half_size, lat - half_size),
    ])
    geometries.append(box)
    gids.append(gid)

# Create a GeoDataFrame - note that it must have a column called "gid" that provides a string ID for each geometry 
gdf = gpd.GeoDataFrame({'gid': gids, 'geometry': geometries}, crs="EPSG:4326")

### Set up our GEE request via a dictionary of parameters
In order to make sure we're grabbing the portions of the ERA5 data that we want, we need to build a dictionary of parameters that defines some of our sampling decisions. In this section, we'll walk through some of these parameters and some possible choices and implications.

#### 1. Select relevant bands
We need to provide the specific ERA5-Land hourly bands we want to sample. If you just want the ELM-required bands, there is a convenient method to request just those: `elm`. You can see the required bands with the following:

In [2]:
print(era5.REQUIRED_RAW_BANDS)

['temperature_2m', 'dewpoint_temperature_2m', 'surface_pressure', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'surface_solar_radiation_downwards_hourly', 'surface_thermal_radiation_downwards_hourly', 'total_precipitation_hourly']


This is the minimal set of bands you'll need to compute the 8 meterologic variables we'll prepare for ELM intake. You'll see when we actually define the `params` dictionary below that there's a shortcut here.

#### 2. Select scale
We can now specify a `gee_scale`. This is the resolution in meters over which GEE should sample the base ERA5-Land hourly data at to perform *spatial averaging* over (multi)polygons. If you're sampling points, this scale should just be `native` which will use the ERA5-Land hourly's native resolution of ~11km, but if you're running bigger polygons or lots of them, you might want to choose a coarser scale to reduce GEE Task runtimes. See [this documentation](https://developers.google.com/earth-engine/guides/scale#:~:text=The%20lowest%20level%20of%20the,within%20a%20256x256%20pixel%20tile.) if you have questions about scale implications. We'll just stick with `native` for this test.


#### 3. Select batch size
We have a parameter that adjusts how big our batch size should be. This parameter is called `gee_years_per_task`, and refers to how many ERA5-Land hourly years each Task sent to GEE should cover. In general, the more geometries you are sampling, the lower you want this number to be. The default of 5 should work OK for a few hundred Point geometries, but if you're doing thousands of Point geometries or larger 2-D geometries you might want to lower it to 1 or 2. Note that there is an optimal number for this parameter in terms of speed of output, but it's basically unknowable. On your end, tt depends on the size of your job (number of geometries and length of time you sample). On the GEE end (unknowable exactly), it depends on current server loads, task prioritization, and memory constraints. **In general, I have found it is better to run more, smaller Tasks on GEE as opposed to larger, fewer ones.** If your Tasks are too large, they also run the risk of *failure*, in which case GEE will retry them (at a cost of lost time to you and potential more *failure*). We set it to 1 in this example to prove that post-processing batching works just fine :) 

#### 4. Specify geometries
We already constructed `gdf` which contains our polygons to sample. We will feed this directly as part of our `params` dictionary. **However**, if you have many polygons, or if they have lots of vertices, this method (using a `GeoDataFrame`) will fail as there is a size limit on geometries that you're able to pass directly from the Python API to GEE. Instead, you will need to upload your shapefile to GEE as an Asset. This is very easy to do; see [this documentation](https://developers.google.com/earth-engine/guides/manage_assets#code-editor_1). Once you've uploaded it, you can now just provide a string to this parameter that represents the path to the asset--for example, `'projects/ee-jonschwenk/assets/AK_temp_20230530'`. Again, here we will just use our gdf. **Remember that your geometries must have a `gid` field containing unique identifiers for each geometry feature.** 

#### 5. Select time range
The ERA5 data we'll be sampling is hourly, and it spans from 1950-01-01 to near-present-day. If you want all available data, use `2200-01-01` as your `end_date`. 

#### 6. Other stuff
See the next block of code for some other intuitive parameters you need to specify related to naming the Tasks and where to store the outputs.

Ok, now we can build our `params` dictionary and spin up some Tasks. I've used comments below to explain what each parameter does in addition to the above explanations. Note that there are more parameters available than what's shown here; this is just a nearly-minimal set you'll need to define.

In [4]:
params = {
    'start_date' : '1950-01-01', # YYYY-MM-DD; no data exists before 1950
    'end_date' : '1955-01-01', # YYYY-MM-DD; we'll sample 5 years here but you can put 2200-01-01 to sample all available data
    'geometries' : gdf, # This can be a GeoDataFrame (as our case), an  ee.FeatureCollection object, or a path to a GEE asset
    'geometry_id_field' : 'gid', # We used "gid" to denote the unique id for each geometry in our GeoDataFrame
    'gee_bands' : 'elm', # Select ELM-required bands; can also select "all" (not recommended) or a list of specific bands you want to sample
    'gee_years_per_task' : 1, # Optional parameter; default is 5. For lots of geometries, you may want to reduce this for smaller GEE Tasks (but more of them)
    'gee_scale' : 'native', # If we provide 2-D geometries, GEE will spatially-average quantities. This sets the scale at which averaging is performed. Choose 'native' for highest resolution (but for large jobs it might take longer). Native is ~11km for ERA5.
    'gdrive_folder' : 'ngee_arctic_sites_example', # Which folder to store on your GDrive; will be created if not exists
    'job_name' : 'all_sites_1x1deg', # This is the name you'll see for each GEE Task
}

# Send the Tasks to GEE! This takes a little while as some time metadata is fetched using getInfo() for GEE.
df_loc = gu.sample_e5lh(params, skip_tasks=False) # skip_tasks=True will generate df_loc without sending the tasks to GEE

Your request will be executed as 5 Tasks in Google Earth Engine.
GEE Export task submitted: all_sites_1x1deg_1950-01-01_1951-01-01
GEE Export task submitted: all_sites_1x1deg_1951-01-01_1952-01-01
GEE Export task submitted: all_sites_1x1deg_1952-01-01_1953-01-01
GEE Export task submitted: all_sites_1x1deg_1953-01-01_1954-01-01
GEE Export task submitted: all_sites_1x1deg_1954-01-01_1955-01-01
All export tasks started. Check Google Drive or Task Status in the Javascript Editor for completion.


> **Tip** If you accidentally sent Tasks to GEE but didn't mean to, you can just call `gu.kill_all_tasks()` which will cancel all Tasks in your queue.

### Now we wait.
We've sent some Tasks (5 of them) to Google Earth Engine. You can check on their state using the [GEE Javascript code editor](http://code.earthengine.google.com) by clicking the `Tasks` tab in the upper-right panel. Eventually it will finish, and your csv will show up where you told GEE to put it: `gdrive_folder/`.

The amount of time you wait totally depends on your job size and GEE's current load. I am able to have 4 Tasks running at once on GEE--any more and they wait for one of the 4 to finish first. GEE controls these things. In general, unless you're running something enormous, it shouldn't take more than a few hours, and can take as little as a few minutes. Also note that the time of day may impact the runtimes, as GEE servers get hit more during 9-5 USA. I ran the above on Saturday and it took 6 minutes to finish all 4 Tasks.

### **Important!** 
Once all Tasks are complete, you must move all the files from your GDrive to a local directory on your machine. Make sure there is nothing else in this folder besides all the files that were exported from this Job. The easiest and fastest way to move files from GDrive to local is with the Google Drive app, but you can also just download from the browser (it's annoying this way because GDrive wants to zip everything which can take awhile if you have lots of files).

## Exporting to ELM-ready formats
At this point, you should have a local directory with all the `csv` files exported from your GEE Tasks. This local directory should have no other files in it besides these. Let's define this now.

In [6]:
csv_directory = Path(r'X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm') # where I've put my downloaded .csv files

### A. Let's try 1 hour resolution; exporting each site as its own set of netCDFS
Once the raw data have been downloaded from GEE, `dapper` will resample it to whatever time resolution you want. This is fast. Currently, there is a bug with E3SM that prevents some 1-hr variables from being timestepped properly. The version of `dapper` at the time of writing this will, by default, provide 30-minute resolution if you ask for 1-hr to sidestep this bug. Once it's fixed, this default behavior will be removed (and you can turn it off now by specifying a keyword argument).

I will put the keyword definitions in the code below rather than writing them out here.

In [9]:
write_directory = csv_directory / 'elm_ready_elm-sites_1hr' # where I want my ELM-formatted netCDFs to go

exp = Exporter(
    adapter=ERA5Adapter(), # Anytime you are writing ERA5, use this Adapter. More will soon be available for custom met files and perhaps other GEE sources (Daymet, GSWP3, etc.)
    csv_directory=csv_directory, # Where dapper can find the raw csvs GEE created and you've downloaded locally
    write_directory=write_directory, # Where dapper should put your exports
    df_loc=df_loc, # This was generated when we called sample_era5() above. Note that you can re-generate it without sending more Tasks to GEE by turning on the skip_tasks parameter in that function.
    dtime_resolution_hrs=1, # Here we select our desired output time resolution. dapper can export basically any time resolution (not sure about weird stuff like 1 minute, but 0.5 hours is fine).
    force_half_hour_for_hourly=True, # I am just showing this becuase of the E3SM bug. If you truly want hourly data, set this to False.
    append_attrs={"note":"hello world"},  # If you want to add information to the exported netCDF files, this is the place to do it. Each netCDF file (variable) will include whatever you specify here.
)

# Now we can run the export
# For output_mode, we have three choices: 'elm-sites', 'elm-combined', and 'elm-grid'. We'll look at examples of the others later; for now, we want to export
# one set of met vars per site, so we choose 'site_dir'.
exp.run(output_mode='elm-sites', pack_scope='global')

Processing file 1 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1950-01-01_1951-01-01.csv
Processing file 2 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1951-01-01_1952-01-01.csv
Processing file 3 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1952-01-01_1953-01-01.csv
Processing file 4 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1953-01-01_1954-01-01.csv
Processing file 5 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1954-01-01_1955-01-01.csv
elm-sites export complete.


#### And you're done!
Look at your `write_directory`. There should be 11 directories, each named the `gid` we specified for our geometries.

And in each of those directories, you should have 7 netCDF files--one for each ELM-required variable. Note that these are following the *coupler bypass* standard set by Dan Riscutto in OLMT. Future development of this repo could allow `DATM_MODE` if it's ever needed. The variables and their required formatting/compression are slightly different between the two.

### B. Now we'll try 3-hour resolution, and we'll put all sites into one set of files
Let's say we want to put all sites into the same netCDF(s), so that there's only one netCDF per variable. We can do this with a simple change in `output_mode`. We will also change the temporal resolution to be 3-hourly instead of hourly.

In [10]:
write_directory = csv_directory / 'elm_ready_elm-combined_3hr' # where I want my ELM-formatted netCDFs to go

exp = Exporter(
    adapter=ERA5Adapter(), 
    csv_directory=csv_directory, # We updated this
    write_directory=write_directory, 
    df_loc=df_loc, 
    dtime_resolution_hrs=3, # We updated this
    append_attrs={"note":"exporting 3hr data for funsies"},  
    )

# Now we can run the export
exp.run(output_mode='elm-combined')

Processing file 1 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1950-01-01_1951-01-01.csv
Processing file 2 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1951-01-01_1952-01-01.csv
Processing file 3 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1952-01-01_1953-01-01.csv
Processing file 4 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1953-01-01_1954-01-01.csv
Processing file 5 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1954-01-01_1955-01-01.csv
elm-combined export complete.


### C. Gridded Exports
`dapper` also supports `output_mode = 'elm-grid'`, which exports one file per variable, but within each exported `netCDF` aligns all observations to a regular grid. Because the data we sampled is averaged over a 1x1 degree cell (not uniformly across a domain), this export method will still work, but it will fill all of the empty space between grid cells with `noData`. 

In [12]:
write_directory = csv_directory / 'elm_ready_gridded_1hr' # where I want my ELM-formatted netCDFs to go

exp = Exporter(
    adapter=ERA5Adapter(), 
    csv_directory=csv_directory, # We updated this
    write_directory=write_directory, 
    df_loc=df_loc, 
    dtime_resolution_hrs=3, # We updated this
    append_attrs={"note":"exporting gridded data example"},  
    )

# Now we can run the export
exp.run(output_mode='elm-grid')

Processing file 1 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1950-01-01_1951-01-01.csv
Processing file 2 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1951-01-01_1952-01-01.csv
Processing file 3 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1952-01-01_1953-01-01.csv
Processing file 4 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1953-01-01_1954-01-01.csv
Processing file 5 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1954-01-01_1955-01-01.csv
elm-grid export complete.


We can look at this [plot]

### D. Non-ELM formats
You can also export your sampled met data as `csv` or `parquet` (note that `pandas` can read `parquet` files which are typically more memory efficient). This is mostly a convenience--you can use `dapper` to sample met data for other, non-ELM applications. As with the previous examples, all you have to do is to change the `output_mode`.

In [13]:
write_directory = csv_directory / 'parquet_1hr' # where I want my ELM-formatted netCDFs to go

exp = Exporter(
    adapter=ERA5Adapter(), 
    csv_directory=csv_directory, # We updated this
    write_directory=write_directory, 
    df_loc=df_loc, 
    dtime_resolution_hrs=1, # We updated this
    append_attrs={"note":"parquet export"},  
    )

# Now we can run the export
exp.run(output_mode='raw-site-parquet') # you can also set output_mode to 'raw-site-csv'

Processing file 1 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1950-01-01_1951-01-01.csv
Processing file 2 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1951-01-01_1952-01-01.csv
Processing file 3 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1952-01-01_1953-01-01.csv
Processing file 4 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1953-01-01_1954-01-01.csv
Processing file 5 of 5: X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\all_sites_1x1deg_1954-01-01_1955-01-01.csv
raw-site-parquet export complete → X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\parquet_1hr\sites_parquet


## Quick validation
The problem with tools like `dapper` is that users may trust the tool too much without verifying their output. There are many ways that things can go wrong in the above processing pipelines. To help mitigate this, `dapper` has some **very basic** functionality to help you visualize some of your results. Here, we'll take a quick look at that.

In [ ]:
from dapper.met import validation
validation.make_quicklooks(exp) # Here, exp is our Exporter object (the last one we defined, which was the parquet export)

quicklooks written to X:\Research\NGEE Arctic\dapper_data\notebook_data\era5-elm\parquet_1hr\quicklooks


If we go to the created `quicklooks` directory, we see a figure for each of the sites in our geometry. Here is an example (the `abs` site):

![Abisko quicklook](../../data/images/abs-quicklook.png)

These plots provide a visual to quickly assess if there are any issues with the sampled data. We can check for continuity (no missing data or weird jumps) and that the ranges of each variable are as expected. If you want to zoom in or change anything about the plots, you'll need to write your own plotting scripts. This tool is just provided for convenience.
